In [0]:
sales = spark.table(
    "agentdb.gold.fact_sales"
)

rolling7 = spark.table(
    "agentdb.features.feature_rolling_7d_sales"
)

rolling30 = spark.table(
    "agentdb.features.feature_rolling_30d_sales"
)

velocity = spark.table(
    "agentdb.features.feature_daily_sales_velocity"
)

variability = spark.table(
    "agentdb.features.feature_demand_variability"
)

inventory = spark.table(
    "agentdb.features.feature_days_of_cover"
)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import *

w = Window.partitionBy(
    "product_key",
    "store_key"
).orderBy(
    "calendar_key"
)

sales_target = (
    sales
    .withColumn(
        "target_sales_qty",
        lead(
            "sales_qty",
            1
        ).over(w)
    )
)

In [0]:
training_dataset = (

    sales_target

    .join(
        rolling7.select(
            "product_key",
            "store_key",
            "calendar_key",
            "rolling_7d_avg",
            "rolling_7d_stddev"
        ),
        [
            "product_key",
            "store_key",
            "calendar_key"
        ]
    )

    .join(
        rolling30.select(
            "product_key",
            "store_key",
            "calendar_key",
            "rolling_30d_avg",
            "rolling_30d_stddev"
        ),
        [
            "product_key",
            "store_key",
            "calendar_key"
        ]
    )

    .join(
        velocity.select(
            "product_key",
            "store_key",
            "calendar_key",
            "sales_velocity"
        ),
        [
            "product_key",
            "store_key",
            "calendar_key"
        ]
    )

    .join(
        variability.select(
            "product_key",
            "store_key",
            "avg_daily_sales",
            "demand_stddev",
            "coefficient_of_variation"
        ),
        [
            "product_key",
            "store_key"
        ]
    )

    .join(
        inventory.select(
            "product_key",
            "store_key",
            "days_of_cover"
        ),
        [
            "product_key",
            "store_key"
        ]
    )

    .filter(
        col("target_sales_qty").isNotNull()
    )
)

In [0]:
(
    training_dataset
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "agentdb.forecasting.forecast_training_dataset"
    )
)

In [0]:
%sql
SELECT * FROM agentdb.forecasting.forecast_training_dataset
LIMIT 10;